# 16 — NRGS common-prefix contrastive setup (validation)

Order-diff query transfer, with the DST frontier line **prefilled by a prefix shared by b and d**, so b and d diverge on the **next token**. That gives a next-token loss for localizing the heads whose query carries progress.

```
SRC   [A, C, B]  prefill {A, B}  ->  c =            frontier C   (query cached at ' =')
DST   [A, B, D]  prefill {A}     ->  b = <prefix>   next: b_tok  (native)
DST*  [A, B, D]  prefill {A, B}  ->  c = <prefix>   next: d_tok  (target distribution)
SRC'  [A, C, B]  prefill {A}     ->  b =            frontier C, B NOT done (control query)
```
**Patch** = selected heads' pre-RoPE query at DST's **last token** (end of prefix) <- SRC's ` =` query.
NRGS predicts b_tok -> d_tok under SRC's query, but not under SRC''s (same "look for C", no "B done").

(b, d, prefix) come from the goal-head localization pivots (`data/head_loc/pairs/<model>.json`; b = destination, d = source). Letters are **sequential by appearance** (`data._seq_letter`).

| § | needs model | what it checks |
|---|---|---|
| 1 | no | bank one-liners pass their unit tests |
| 2 | no | pivot pool, instance yield, rendered roles |
| 3 | yes | divergence gate in the NRGS context (cached) |
| 4 | yes | greedy line continuations per role (eyeball) |
| 5 | yes | one example end to end: highlighted prompts, top-10s, patched run, IE |
| 6 | yes | baseline over a stratified sample: goal / control heads, SRC vs SRC', flip / Δmargin / IE (cached) |

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, json, collections
while not os.path.isdir('src') and os.getcwd() != '/':
    os.chdir('..')
sys.path.append('.')

# ============================= KNOBS =============================
MODEL             = 'Qwen/Qwen2.5-Coder-32B-Instruct'
CUDA              = '0,1'            # set free GPU(s) BEFORE §3 loads the model
VARIANT           = 'order_diff'     # 'order_diff' | 'order_matched' | 'c_in_dst'
DATATYPES         = ['str', 'int', 'list[int]']
N_PER_PIVOT       = 2                # (A, C) orderings per (b, d, prefix) pivot
DISTINCT_FAMILIES = True             # A, C avoid near-duplicates of every other role
SEED              = 0
SRC_LANG, DST_LANG = 'python', 'python'
GOAL_HEADS = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json'
CTRL_HEADS = 'head_sets/Qwen2.5-Coder-32B-Instruct/control/control250__laterhalf.json'
N_BASELINE = 64                      # gate-passed instances for the §6 baseline (stratified)
OV_BOOST   = 2.0                     # x the patched heads' OV write at DST's last token (paper q-transfer: 2.0; 1.0 = query patch only)
# ================================================================

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.markup import escape
console = Console(highlight=False)
import src.diff.nrgs_prefix as N
from src.utils import env_utils
CACHE = os.path.join(env_utils.DEFAULT_RESULTS_DIR, 'notebook_caches', 'nrgs_prefix', MODEL.split('/')[-1])
os.makedirs(CACHE, exist_ok=True)

## §1 — bank (no model)

In [2]:
rows = N.validate_bank()
t = Table('datatype', 'predicate', 'one-liner', 'tests', 'fail', title='goal-head localization bank')
for r in rows:
    t.add_row(r['datatype'], r['id'], escape(r['impl']), str(r['n_tests']),
              '[red]%d[/red]' % r['n_fail'] if r['n_fail'] else '0')
console.print(t)
assert not any(r['n_fail'] for r in rows), 'fix the failing one-liners before going on'

                                          goal-head localization bank                                           
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━┓
┃ datatype ┃ predicate              ┃ one-liner                                                 ┃ tests ┃ fail ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━┩
│ list     │ all_even               │ all(x % 2 == 0 for x in lst)                              │ 6     │ 0    │
│ list     │ all_positive           │ all(x > 0 for x in lst)                                   │ 6     │ 0    │
│ list     │ all_same               │ len(set(lst)) <= 1                                        │ 6     │ 0    │
│ str      │ ends_with_vowel        │ bool(s) and s[-1].lower() in 'aeiou'                      │ 6     │ 0    │
│ list     │ has_duplicates         │ len(set(lst)) != len(lst)                                 │ 5     │ 0    │
│ str      │ has_spaces             │ ' ' in s                                                  │ 6     │ 0    │
│ str      │ is_digit_string        │ s.isdigit()                                               │ 6     │ 0    │
│ int      │ is_divisible_by_three  │ n % 3 == 0                                                │ 7     │ 0    │
│ int      │ is_even                │ n % 2 == 0                                                │ 6     │ 0    │
│ str      │ is_lower               │ s == s.lower()                                            │ 6     │ 0    │
│ int      │ is_odd                 │ n % 2 != 0                                                │ 6     │ 0    │
│ str      │ is_palindrome          │ s == s[::-1]                                              │ 7     │ 0    │
│ int      │ is_perfect_square      │ n >= 0 and int(n ** 0.5) ** 2 == n                        │ 7     │ 0    │
│ int      │ is_positive            │ n > 0                                                     │ 6     │ 0    │
│ int      │ is_power_of_two        │ n > 0 and (n & (n - 1)) == 0                              │ 7     │ 0    │
│ int      │ is_prime               │ n > 1 and all(n % i for i in range(2, int(n ** 0.5) + 1)) │ 7     │ 0    │
│ list     │ is_sorted              │ all(lst[i] <= lst[i + 1] for i in range(len(lst) - 1))    │ 5     │ 0    │
│ list     │ is_strictly_decreasing │ all(lst[i] > lst[i + 1] for i in range(len(lst) - 1))     │ 6     │ 0    │
│ list     │ is_strictly_sorted     │ all(lst[i] < lst[i + 1] for i in range(len(lst) - 1))     │ 6     │ 0    │
│ str      │ is_upper               │ s == s.upper()                                            │ 4     │ 0    │
│ str      │ starts_with_vowel      │ bool(s) and s[0].lower() in 'aeiou'                       │ 6     │ 0    │
└──────────┴────────────────────────┴───────────────────────────────────────────────────────────┴───────┴──────┘

## §2 — pivot pool, instances, rendered roles (no model)

In [3]:
PIVOTS = N.load_pivots(MODEL, datatypes=DATATYPES, core_only=True)
INSTS  = N.build_instances(PIVOTS, variant=VARIANT, n_per_pivot=N_PER_PIVOT, seed=SEED,
                           distinct_families=DISTINCT_FAMILIES, src_lang=SRC_LANG, dst_lang=DST_LANG)
TRAIN, VAL = N.split_by_pair(INSTS, val_frac=0.2, seed=SEED)
t = Table('datatype', 'pivots', 'instances', 'train', 'val', 'val pairs', title=f'{VARIANT} · core prefixes')
for dt in DATATYPES:
    f = lambda xs: [x for x in xs if x.datatype == dt]
    t.add_row(dt, str(sum(p.datatype == dt for p in PIVOTS)), str(len(f(INSTS))), str(len(f(TRAIN))),
              str(len(f(VAL))), str(len({N.pair_id(x) for x in f(VAL)})))
console.print(t)

                order_diff · core prefixes                 
┏━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━┳━━━━━┳━━━━━━━━━━━┓
┃ datatype ┃ pivots ┃ instances ┃ train ┃ val ┃ val pairs ┃
┡━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━╇━━━━━╇━━━━━━━━━━━┩
│ str      │ 108    │ 216       │ 170   │ 46  │ 4         │
│ int      │ 80     │ 160       │ 126   │ 34  │ 4         │
│ list     │ 132    │ 264       │ 226   │ 38  │ 3         │
└──────────┴────────┴───────────┴───────┴─────┴───────────┘

In [4]:
# pick an instance to eyeball (b, d, prefix); falls back to the first instance
SHOW_B, SHOW_D, SHOW_PREFIX = 'is_lower', 'is_palindrome', 's == s'
x = next((i for i in INSTS if (i.pivot.b, i.pivot.d, i.pivot.prefix) == (SHOW_B, SHOW_D, SHOW_PREFIX)), INSTS[0])
console.print(f'[bold]{escape(x.key)}[/bold]   head_loc tokens: b→[green]{escape(repr(x.pivot.b_tok))}[/green]  d→[magenta]{escape(repr(x.pivot.d_tok))}[/magenta]')
for role, (q, code) in x.parts().items():
    console.print(Panel(escape(q + code), title=role, title_align='left', border_style='grey42'))

order_diff|str|A=starts_with_vowel|B=is_lower|C=is_digit_string|D=is_palindrome|prefix=s == s|python-python   
head_loc tokens: b→'.lower'  d→'[::-'

╭─ src ───────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Write a Python program that implements a function `func` which, given a string `s`, computes:                   │
│ (1) whether the string starts with a vowel (case-insensitive)                                                   │
│ (2) whether every character is a digit                                                                          │
│ (3) whether all alphabetic characters are lowercase                                                             │
│ <code>                                                                                                          │
│ def func(s: str):                                                                                               │
│     a = bool(s) and s[0].lower() in 'aeiou'                                                                     │
│     b = s == s.lower()                                                                                          │
│     c =                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ dst ───────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Write a Python program that implements a function `func` which, given a string `s`, computes:                   │
│ (1) whether the string starts with a vowel (case-insensitive)                                                   │
│ (2) whether all alphabetic characters are lowercase                                                             │
│ (3) whether the string is a palindrome                                                                          │
│ <code>                                                                                                          │
│ def func(s: str):                                                                                               │
│     a = bool(s) and s[0].lower() in 'aeiou'                                                                     │
│     b = s == s                                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ dst_star ──────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Write a Python program that implements a function `func` which, given a string `s`, computes:                   │
│ (1) whether the string starts with a vowel (case-insensitive)                                                   │
│ (2) whether all alphabetic characters are lowercase                                                             │
│ (3) whether the string is a palindrome                                                                          │
│ <code>                                                                                                          │
│ def func(s: str):                                                                                               │
│     a = bool(s) and s[0].lower() in 'aeiou'                                                                     │
│     b = s == s.lower()                                                                                          │
│     c = s == s                                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ src_ctrl ──────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Write a Python program that implements a function `func` which, given a string `s`, computes:                   │
│ (1) whether the string starts with a vowel (case-insensitive)                                                   │
│ (2) whether every character is a digit                                                                          │
│ (3) whether all alphabetic characters are lowercase                                                             │
│ <code>                                                                                                          │
│ def func(s: str):                                                                                               │
│     a = bool(s) and s[0].lower() in 'aeiou'                                                                     │
│     b =                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## §3 — divergence gate (model; cached)
`b_id`/`d_id` = the model's top-1 after the prefix in DST / DST* (these become the loss targets). `passed` also requires them to equal the head_loc tokens.

In [5]:
os.environ['CUDA_VISIBLE_DEVICES'] = CUDA
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
import torch
import src.diff.functional as FUNC
ctx = FUNC.setup(MODEL); mt = ctx.mt

2026-09-29 16:38:51 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


In [6]:
from tqdm.auto import tqdm
GATE_PATH = os.path.join(CACHE, f'gate__{VARIANT}__{SRC_LANG}-{DST_LANG}.json')
done = {r['key']: r for r in json.load(open(GATE_PATH))} if os.path.exists(GATE_PATH) else {}
todo = [x for x in INSTS if x.key not in done]
print(f'{len(INSTS)} instances · {len(done)} cached · {len(todo)} to gate -> {GATE_PATH}')
for s in tqdm(range(0, len(todo), 32), desc='gate'):
    for r in N.gate(mt, ctx, todo[s:s + 32], batch_size=8):
        done[r['key']] = r
    json.dump(list(done.values()), open(GATE_PATH, 'w'))       # checkpoint every chunk
GATE = {x.key: done[x.key] for x in INSTS}

t = Table('datatype', 'n', 'diverge', 'match b', 'match d', 'passed', title='NRGS-context gate')
for dt in DATATYPES + ['ALL']:
    g = [r for r in GATE.values() if dt == 'ALL' or r['datatype'] == dt]
    pct = lambda k: f"{sum(r[k] for r in g) / max(1, len(g)):.2f}"
    t.add_row(dt, str(len(g)), pct('diverge'), pct('match_b'), pct('match_d'), pct('passed'))
console.print(t)

640 instances · 640 cached · 0 to gate -> /disk/u/arnab/Codes/Projects/goals/results/notebook_caches/nrgs_prefix/Qwen2.5-Coder-32B-Instruct/gate__order_diff__python-python.json


gate: 0it [00:00, ?it/s]

                    NRGS-context gate                    
┏━━━━━━━━━━┳━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┓
┃ datatype ┃ n   ┃ diverge ┃ match b ┃ match d ┃ passed ┃
┡━━━━━━━━━━╇━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━┩
│ str      │ 216 │ 0.88    │ 0.81    │ 0.88    │ 0.70   │
│ int      │ 160 │ 0.79    │ 0.66    │ 0.64    │ 0.49   │
│ list     │ 264 │ 0.98    │ 0.94    │ 0.94    │ 0.88   │
│ ALL      │ 640 │ 0.90    │ 0.83    │ 0.84    │ 0.72   │
└──────────┴─────┴─────────┴─────────┴─────────┴────────┘

In [7]:
# where the gate fails: prefix, head_loc tokens vs the model's top-1 in DST / DST*
fails = [r for r in GATE.values() if not r['passed']]
t = Table('datatype', 'b → d', 'prefix', 'DST top-1 (hl b)', 'DST* top-1 (hl d)', 'diverge', title=f'{len(fails)} gate failures (first 25)')
for r in fails[:25]:
    x = next(i for i in INSTS if i.key == r['key'])
    t.add_row(r['datatype'], f"{r['b']} → {r['d']}", escape(repr(r['prefix'])),
              f"{escape(repr(r['top1_dst']))} ({escape(repr(x.pivot.b_tok))})",
              f"{escape(repr(r['top1_dst_star']))} ({escape(repr(x.pivot.d_tok))})", str(r['diverge']))
console.print(t)

                                           178 gate failures (first 25)                                            
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ datatype ┃ b → d                                 ┃ prefix      ┃ DST top-1 (hl b) ┃ DST* top-1 (hl d) ┃ diverge ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ list     │ has_duplicates → all_even             │ 'len(lst)'  │ ' !=' (' !=')    │ ' >' (' ==')      │ True    │
│ list     │ has_duplicates → all_even             │ 'len(lst)'  │ ' !=' (' !=')    │ ' >' (' ==')      │ True    │
│ list     │ has_duplicates → all_even             │ 'not'       │ ' all' (' len')  │ ' any' (' any')   │ True    │
│ list     │ all_even → has_duplicates             │ 'len(lst)'  │ ' >' (' ==')     │ ' !=' (' !=')     │ True    │
│ list     │ all_even → has_duplicates             │ 'len(lst)'  │ ' >' (' ==')     │ ' !=' (' !=')     │ True    │
│ list     │ all_even → has_duplicates             │ 'not'       │ ' any' (' any')  │ ' all' (' len')   │ True    │
│ list     │ all_even → has_duplicates             │ 'not'       │ ' any' (' any')  │ ' all' (' len')   │ True    │
│ list     │ all_even → is_strictly_decreasing     │ 'not any(x' │ ' %' (' %')      │ '1' (' <=')       │ True    │
│ list     │ all_even → is_strictly_decreasing     │ 'not any(x' │ ' %' (' %')      │ '1' (' <=')       │ True    │
│ list     │ all_same → all_positive               │ 'not'       │ ' any' (' lst')  │ ' any' (' any')   │ False   │
│ list     │ has_duplicates → all_positive         │ 'not'       │ ' all' (' len')  │ ' any' (' any')   │ True    │
│ list     │ all_positive → has_duplicates         │ 'not'       │ ' any' (' any')  │ ' all' (' len')   │ True    │
│ list     │ all_positive → has_duplicates         │ 'not'       │ ' any' (' any')  │ ' all' (' len')   │ True    │
│ list     │ all_same → is_sorted                  │ 'lst'       │ ' ==' ('[')      │ ' ==' (' ==')     │ False   │
│ list     │ all_same → is_sorted                  │ 'lst'       │ ' ==' ('[')      │ ' ==' (' ==')     │ False   │
│ list     │ all_same → is_strictly_decreasing     │ 'lst'       │ ' ==' ('[')      │ ' ==' (' ==')     │ False   │
│ list     │ all_same → is_strictly_sorted         │ 'lst'       │ ' ==' ('[')      │ ' ==' (' ==')     │ False   │
│ list     │ is_sorted → has_duplicates            │ 'not'       │ ' any' (' any')  │ ' all' (' len')   │ True    │
│ list     │ is_sorted → has_duplicates            │ 'not'       │ ' any' (' any')  │ ' all' (' len')   │ True    │
│ list     │ is_strictly_decreasing →              │ 'len(lst)'  │ ' ==' (' <')     │ ' !=' (' !=')     │ True    │
│          │ has_duplicates                        │             │                  │                   │         │
│ list     │ is_strictly_decreasing →              │ 'len(lst)'  │ ' ==' (' <')     │ ' !=' (' !=')     │ True    │
│          │ has_duplicates                        │             │                  │                   │         │
│ list     │ is_strictly_decreasing →              │ 'not'       │ ' any' (' any')  │ ' all' (' len')   │ True    │
│          │ has_duplicates                        │             │                  │                   │         │
│ list     │ is_strictly_decreasing →              │ 'not'       │ ' any' (' any')  │ ' all' (' len')   │ True    │
│          │ has_duplicates                        │             │                  │                   │         │
│ list     │ has_duplicates →                      │ 'not'       │ ' all' (' len')  │ ' any' (' any')   │ True    │
│          │ is_strictly_decreasing                │             │                  │                   │         │
│ list     │ is_strictly_sorted → has_duplicates   │ 'len(lst)'  │ ' ==' (' <')     │ ' !=' (' !=')     │ True    │
└──────────┴───────────────────────────────────────┴────

## §4 — greedy line continuations (model; eyeball)
Expect: SRC and SRC' implement **C**, DST continues **b**, DST* continues **d**.

In [8]:
from src.goals.goalscope import greedy_continue
from src.tokens import prepare_input
N_SHOW = 4
COLOR = {'src': 'cyan', 'src_ctrl': 'cyan', 'dst': 'green', 'dst_star': 'magenta'}

@torch.no_grad()
def line_after(prompt, max_new=32):
    inp = prepare_input(prompts=prompt, tokenizer=mt)
    return greedy_continue(mt, inp['input_ids'], inp['attention_mask'], max_new).split('\n')[0]

for x in [i for i in INSTS if GATE[i.key]['passed']][:N_SHOW]:
    ps, parts, bank = x.prompts(ctx), x.parts(), N.BANK[x.datatype]
    body = [f"[bold]{escape(x.key)}[/bold]",
            f"C = {escape(bank[x.c].description)}   b = {escape(bank[x.pivot.b].description)}   d = {escape(bank[x.pivot.d].description)}"]
    for role in parts:
        last = parts[role][1].split('\n')[-1]
        body.append(f"[grey58]{role:9s}[/grey58] {escape(last)}[{COLOR[role]}]{escape(line_after(ps[role]))}[/{COLOR[role]}]")
    console.print(Panel('\n'.join(body), border_style='grey42'))

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ order_diff|list[int]|A=all_positive|B=all_same|C=is_sorted|D=all_even|prefix=all(x|python-python                │
│ C = whether the list is sorted in non-decreasing order   b = whether all elements are equal   d = whether every │
│ element is even                                                                                                 │
│ src           c = lst == sorted(lst)                                                                            │
│ dst           b = all(x == lst[0] for x in lst)                                                                 │
│ dst_star      c = all(x % 2 == 0 for x in lst)                                                                  │
│ src_ctrl      b = all(lst[i] <= lst[i + 1] for i in range(len(lst) - 1))                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ order_diff|list[int]|A=is_strictly_decreasing|B=all_same|C=all_positive|D=all_even|prefix=all(x|python-python   │
│ C = whether every element is positive   b = whether all elements are equal   d = whether every element is even  │
│ src           c = all(x > 0 for x in lst)                                                                       │
│ dst           b = all(x == lst[0] for x in lst)                                                                 │
│ dst_star      c = all(x % 2 == 0 for x in lst)                                                                  │
│ src_ctrl      b = all(x > 0 for x in lst)                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ order_diff|list[int]|A=all_positive|B=all_same|C=is_strictly_sorted|D=all_even|prefix=all(lst[i]|python-python  │
│ C = whether the list is strictly increasing   b = whether all elements are equal   d = whether every element is │
│ even                                                                                                            │
│ src           c = all(lst[i] < lst[i+1] for i in range(len(lst)-1))                                             │
│ dst           b = all(lst[i] == lst[0] for i in range(1, len(lst))) if lst else True                            │
│ dst_star      c = all(lst[i] % 2 == 0 for i in range(len(lst)))                                                 │
│ src_ctrl      b = all(lst[i] < lst[i + 1] for i in range(len(lst) - 1))                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ order_diff|list[int]|A=is_strictly_sorted|B=all_same|C=has_duplicates|D=all_even|prefix=all(lst[i]|python-pytho │
│ n                                                                                                               │
│ C = whether any element appears more than once   b = whether all elements are equal   d = whether every element │
│ is even                                                                                                         │
│ src           c = len(set(lst)) != len(lst)                                                                     │
│ dst           b = all(lst[i] == lst[i + 1] for i in range(len(lst) - 1))                                        │
│ dst_star      c = all(lst[i] % 2 == 0 for i in range(len(lst)))                                                 │
│ src_ctrl      b = len(lst) != len(set(lst))                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## §5 — one example, end to end (model)
For a single gate-passed example: the full SRC / DST / DST* prompts with the relevant token highlighted, each run's top-10 next tokens, then the patched run.
- **SRC** — highlighted token = where the heads' pre-RoPE **query is cached** (the ` =` pivot).
- **DST** — highlighted token = the **pivot**: the last prefix token, where the cached query is written and where b and d diverge.
- **DST\*** — highlighted token = where the **target distribution** is read.

Patched effect on the target token t = d's divergence token (logits, not log-probs):
`IE = (logit M(dst | <- q_src)[t] - logit M(dst)[t]) / (logit M(dst*)[t] - logit M(dst)[t])` — 1 = the patch moves t as far as the "B done" counterfactual, 0 = no effect.
Rows for b / d are coloured green / magenta. The patched run also scales the patched heads' **OV write** at that token by `OV_BOOST` (`boost_head_ov`, stacked on the query patch exactly as in the paper's `--alphas 2` runs); the summary table reports both α=1 and α=`OV_BOOST`.

In [9]:
from rich.text import Text
from src.functional import interpret_logits
from src.tokens import prepare_input

# ── pick ONE gate-passed example (falls back to the first passed one) ──
EX_B, EX_D, EX_PREFIX = 'is_lower', 'is_palindrome', 's == s'
EX_HEADS, EX_SRC_ROLE = 'goal250', 'src'          # patch with these heads; query taken from this role

GOAL = [(int(l), int(h)) for l, h in json.load(open(GOAL_HEADS))]
CTRL = [(int(l), int(h)) for l, h in json.load(open(CTRL_HEADS))]
HEADSETS = {'goal250': GOAL, 'ctrl250': CTRL}
SRC_ROLES = ['src', 'src_ctrl'] if VARIANT == 'order_diff' else ['src']

_passed = [x for x in INSTS if GATE[x.key]['passed']]
EX = next((x for x in _passed if (x.pivot.b, x.pivot.d, x.pivot.prefix) == (EX_B, EX_D, EX_PREFIX)), _passed[0])
B_ID, D_ID = GATE[EX.key]['b_id'], GATE[EX.key]['d_id']
PS = EX.prompts(ctx)
bank = N.BANK[EX.datatype]
console.print(f"[bold]{escape(EX.key)}[/bold]")
console.print(f"A={escape(bank[EX.a].description)} · B={escape(bank[EX.pivot.b].description)} · "
              f"C={escape(bank[EX.c].description)} · D={escape(bank[EX.pivot.d].description)}")
console.print(f"b token = [green]{escape(repr(mt.tokenizer.decode([B_ID])))}[/green] (id {B_ID})   "
              f"d token = [magenta]{escape(repr(mt.tokenizer.decode([D_ID])))}[/magenta] (id {D_ID})")

def show_marked(prompt, title, color):
    '''Full prompt with its LAST token highlighted (the position that matters for this role).'''
    off = prepare_input(prompts=prompt, tokenizer=mt, return_offsets_mapping=True)['offset_mapping'][0].tolist()
    s, e = off[-1]
    t = Text(prompt[:s]); t.append(prompt[s:e], style=f'bold black on {color}'); t.append(prompt[e:])
    console.print(Panel(t, title=f'{title} · token #{len(off) - 1} = {prompt[s:e]!r}', title_align='left',
                        border_style=color))

def topk_table(logits, title, k=10):
    '''predict_next_token-style top-k (rank / token / id / prob / logit), plus b and d if outside the top-k.'''
    preds, track = interpret_logits(tokenizer=mt, logits=logits, k=k, interested_tokens=[B_ID, D_ID])
    col = {B_ID: 'green', D_ID: 'magenta'}
    t = Table('rank', 'token', 'id', 'prob', 'logit', title=title)
    for i, pt in enumerate(preds, 1):
        t.add_row(str(i), escape(repr(pt.token)), str(pt.token_id), f'{pt.prob:.3f}', f'{pt.logit:.2f}',
                  style=col.get(pt.token_id))
    for tid, lab in ((B_ID, 'b'), (D_ID, 'd')):
        rank, pt = track[tid]
        if rank > k:
            t.add_row(str(rank), escape(repr(pt.token)) + f'  ({lab})', str(tid), f'{pt.prob:.2e}', f'{pt.logit:.2f}',
                      style=col[tid])
    return t

order_diff|str|A=starts_with_vowel|B=is_lower|C=is_digit_string|D=is_palindrome|prefix=s == s|python-python

A=whether the string starts with a vowel (case-insensitive) · B=whether all alphabetic characters are lowercase · 
C=whether every character is a digit · D=whether the string is a palindrome

b token = '.lower' (id 18073)   d token = '[::-' (id 62154)

In [10]:
# ── the three prompts, highlighted, with their top-10 next tokens (single-prompt forwards, no padding) ──
assert PS['dst'].endswith(EX.pivot.prefix), 'DST must end with the shared prefix'
assert PS['src'].rstrip().endswith('='), 'SRC must end at the ` =` pivot'
L_SRC = N.last_token_logits(mt, [PS['src']])[0]
L_DST = N.last_token_logits(mt, [PS['dst']])[0]
L_STAR = N.last_token_logits(mt, [PS['dst_star']])[0]

show_marked(PS['src'], 'SRC  — query cached here', 'cyan')
console.print(topk_table(L_SRC, 'SRC next token (should start implementing C)'))
show_marked(PS['dst'], 'DST  — pivot: cached query written here; b vs d diverge next', 'green')
console.print(topk_table(L_DST, 'DST next token (clean; expect b)'))
show_marked(PS['dst_star'], 'DST* — target distribution read here', 'magenta')
console.print(topk_table(L_STAR, 'DST* next token (expect d)'))

╭─ SRC  — query cached here · token #139 = ' =' ──────────────────────────────────────────────────────────────────╮
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Write a Python program that implements a function `func` which, given a string `s`, computes:                   │
│ (1) whether the string starts with a vowel (case-insensitive)                                                   │
│ (2) whether every character is a digit                                                                          │
│ (3) whether all alphabetic characters are lowercase                                                             │
│ <|im_end|>                                                                                                      │
│ <|im_start|>assistant                                                                                           │
│ <code>                                                                                                          │
│ def func(s: str):                                                                                               │
│     a = bool(s) and s[0].lower() in 'aeiou'                                                                     │
│     b = s == s.lower()                                                                                          │
│     c =                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

    SRC next token (should start implementing C)    
┏━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━┓
┃ rank  ┃ token         ┃ id    ┃ prob     ┃ logit ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━┩
│ 1     │ ' s'          │ 274   │ 0.807    │ 23.25 │
│ 2     │ ' all'        │ 678   │ 0.159    │ 21.62 │
│ 3     │ ' bool'       │ 1807  │ 0.028    │ 19.88 │
│ 4     │ ' b'          │ 293   │ 0.004    │ 17.88 │
│ 5     │ ' not'        │ 537   │ 0.002    │ 17.00 │
│ 6     │ ' len'        │ 2422  │ 0.000    │ 14.94 │
│ 7     │ " ''."        │ 42863 │ 0.000    │ 14.56 │
│ 8     │ ' any'        │ 894   │ 0.000    │ 14.44 │
│ 9     │ " '"          │ 364   │ 0.000    │ 14.25 │
│ 10    │ ' ('          │ 320   │ 0.000    │ 14.06 │
│ 6915  │ '.lower'  (b) │ 18073 │ 3.10e-10 │ 1.57  │
│ 40715 │ '[::-'  (d)   │ 62154 │ 2.28e-11 │ -1.04 │
└───────┴───────────────┴───────┴──────────┴───────┘

╭─ DST  — pivot: cached query written here; b vs d diverge next · token #134 = ' s' ──────────────────────────────╮
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Write a Python program that implements a function `func` which, given a string `s`, computes:                   │
│ (1) whether the string starts with a vowel (case-insensitive)                                                   │
│ (2) whether all alphabetic characters are lowercase                                                             │
│ (3) whether the string is a palindrome                                                                          │
│ <|im_end|>                                                                                                      │
│ <|im_start|>assistant                                                                                           │
│ <code>                                                                                                          │
│ def func(s: str):                                                                                               │
│     a = bool(s) and s[0].lower() in 'aeiou'                                                                     │
│     b = s == s                                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

          DST next token (clean; expect b)          
┏━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━┓
┃ rank ┃ token          ┃ id    ┃ prob     ┃ logit ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━┩
│ 1    │ '.lower'       │ 18073 │ 1.000    │ 29.25 │
│ 2    │ '[:'           │ 3447  │ 0.000    │ 16.75 │
│ 3    │ ' lower'       │ 4722  │ 0.000    │ 16.50 │
│ 4    │ '['            │ 58    │ 0.000    │ 16.12 │
│ 5    │ 'lower'        │ 14772 │ 0.000    │ 15.75 │
│ 6    │ '.toLowerCase' │ 16924 │ 0.000    │ 15.62 │
│ 7    │ '.ascii'       │ 88897 │ 0.000    │ 15.38 │
│ 8    │ '.replace'     │ 6980  │ 0.000    │ 15.25 │
│ 9    │ '.upper'       │ 33664 │ 0.000    │ 15.00 │
│ 10   │ '.case'        │ 68687 │ 0.000    │ 14.88 │
│ 20   │ '[::-'  (d)    │ 62154 │ 1.86e-07 │ 13.75 │
└──────┴────────────────┴───────┴──────────┴───────┘

╭─ DST* — target distribution read here · token #142 = ' s' ──────────────────────────────────────────────────────╮
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Write a Python program that implements a function `func` which, given a string `s`, computes:                   │
│ (1) whether the string starts with a vowel (case-insensitive)                                                   │
│ (2) whether all alphabetic characters are lowercase                                                             │
│ (3) whether the string is a palindrome                                                                          │
│ <|im_end|>                                                                                                      │
│ <|im_start|>assistant                                                                                           │
│ <code>                                                                                                          │
│ def func(s: str):                                                                                               │
│     a = bool(s) and s[0].lower() in 'aeiou'                                                                     │
│     b = s == s.lower()                                                                                          │
│     c = s == s                                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

          DST* next token (expect d)          
┏━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃ rank ┃ token       ┃ id    ┃ prob  ┃ logit ┃
┡━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 1    │ '[::-'      │ 62154 │ 1.000 │ 29.00 │
│ 2    │ '.lower'    │ 18073 │ 0.000 │ 18.50 │
│ 3    │ '[:'        │ 3447  │ 0.000 │ 18.12 │
│ 4    │ '['         │ 58    │ 0.000 │ 17.75 │
│ 5    │ '[-'        │ 7609  │ 0.000 │ 16.88 │
│ 6    │ '[len'      │ 24693 │ 0.000 │ 15.62 │
│ 7    │ '.replace'  │ 6980  │ 0.000 │ 14.75 │
│ 8    │ '.reverse'  │ 32081 │ 0.000 │ 14.56 │
│ 9    │ '[:]'       │ 74294 │ 0.000 │ 13.62 │
│ 10   │ ' reversed' │ 27437 │ 0.000 │ 13.56 │
└──────┴─────────────┴───────┴───────┴───────┘

In [13]:
# ── patched run: DST with EX_HEADS' query at its last token <- EX_SRC_ROLE's ` =` query, OV write x OV_BOOST ──
OV_BOOST = 1.0
L_PAT = N.q_patch_logits(mt, ctx, EX, HEADSETS[EX_HEADS], src_role=EX_SRC_ROLE, ov_boost=OV_BOOST)
console.print(topk_table(L_PAT, f'patched: DST | <- q_{EX_SRC_ROLE}  ({EX_HEADS}, OV x{OV_BOOST:g})'))
ie = N.indirect_effect(L_PAT, L_DST, L_STAR, D_ID)
console.print(f"logit[d]:  dst = {float(L_DST[D_ID]):.2f}   dst* = {float(L_STAR[D_ID]):.2f}   "
              f"patched = {float(L_PAT[D_ID]):.2f}   ->  IE = ({float(L_PAT[D_ID]):.2f} - {float(L_DST[D_ID]):.2f}) / "
              f"({float(L_STAR[D_ID]):.2f} - {float(L_DST[D_ID]):.2f}) = [bold]{ie:+.3f}[/bold]")

# same example: every head set x query source (SRC' = same frontier C, B not done) x OV boost
ALPHAS = sorted({1.0, float(OV_BOOST)})
t = Table('heads', 'query from', 'OV x', 'top-1', 'logit[b] Δ', 'logit[d] Δ', 'IE (d)',
          title='all conditions · this example')
for hname, hs in HEADSETS.items():
    for role in SRC_ROLES:
        for a in ALPHAS:
            lp = (L_PAT if (hname, role, a) == (EX_HEADS, EX_SRC_ROLE, float(OV_BOOST))
                  else N.q_patch_logits(mt, ctx, EX, hs, src_role=role, ov_boost=a))
            t.add_row(hname, role, f'{a:g}', escape(repr(mt.tokenizer.decode([int(lp.argmax())]))),
                      f'{float(lp[B_ID] - L_DST[B_ID]):+.2f}', f'{float(lp[D_ID] - L_DST[D_ID]):+.2f}',
                      f'{N.indirect_effect(lp, L_DST, L_STAR, D_ID):+.3f}')
console.print(t)

  patched: DST | <- q_src  (goal250, OV x1)  
┏━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃ rank ┃ token      ┃ id    ┃ prob  ┃ logit ┃
┡━━━━━━╇━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 1    │ '.lower'   │ 18073 │ 1.000 │ 25.62 │
│ 2    │ '[:'       │ 3447  │ 0.000 │ 16.88 │
│ 3    │ '.upper'   │ 33664 │ 0.000 │ 16.75 │
│ 4    │ '.replace' │ 6980  │ 0.000 │ 15.00 │
│ 5    │ '.l'       │ 918   │ 0.000 │ 15.00 │
│ 6    │ '['        │ 58    │ 0.000 │ 14.94 │
│ 7    │ '[::-'     │ 62154 │ 0.000 │ 14.06 │
│ 8    │ '.is'      │ 2079  │ 0.000 │ 14.06 │
│ 9    │ ' lower'   │ 4722  │ 0.000 │ 14.00 │
│ 10   │ '.al'      │ 12168 │ 0.000 │ 13.94 │
└──────┴────────────┴───────┴───────┴───────┘

logit:  dst = 13.75   dst* = 29.00   patched = 14.06   ->  IE = (14.06 - 13.75) / (29.00 - 13.75) = +0.020

                     all conditions · this example                     
┏━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┓
┃ heads   ┃ query from ┃ OV x ┃ top-1    ┃ logit Δ ┃ logit Δ ┃ IE (d) ┃
┡━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━┩
│ goal250 │ src        │ 1    │ '.lower' │ -3.62   │ +0.31   │ +0.020 │
│ goal250 │ src_ctrl   │ 1    │ '.lower' │ -5.38   │ -0.38   │ -0.025 │
│ ctrl250 │ src        │ 1    │ '.lower' │ -1.12   │ -0.56   │ -0.037 │
│ ctrl250 │ src_ctrl   │ 1    │ '.lower' │ -2.00   │ -0.81   │ -0.053 │
└─────────┴────────────┴──────┴──────────┴─────────┴─────────┴────────┘

## §6 — baseline: hard q-patch over a stratified sample (model; cached)
SRC's (or SRC''s) ` =` query replaces the heads' query at DST's last token; one forward per condition. The sample is **stratified**: balanced across datatypes and spread round-robin over (b, d) pairs (`N.stratified_sample`). Tables count **only the current sample's** rows.
`flip` = top-1 becomes d_id · `Δmargin` = change in logp(d) − logp(b) vs clean DST · `IE` = normalized indirect effect on d's logit (as in §5; the median is robust when the denominator is small). Runs at the current `OV_BOOST`; rows are cached per α, and the tables show every α cached for this sample (run once at 1.0 and once at 2.0 to compare). NRGS predicts a **SRC > SRC'** gap for the heads that carry progress.

In [12]:
import statistics
GOAL = [(int(l), int(h)) for l, h in json.load(open(GOAL_HEADS))]
CTRL = [(int(l), int(h)) for l, h in json.load(open(CTRL_HEADS))]
HEADSETS = {'goal250': GOAL, 'ctrl250': CTRL}
SRC_ROLES = ['src', 'src_ctrl'] if VARIANT == 'order_diff' else ['src']

SAMPLE = N.stratified_sample([x for x in INSTS if GATE[x.key]['passed']], N_BASELINE, seed=SEED)
KEYS = {x.key for x in SAMPLE}
console.print(f"sample n={len(SAMPLE)} · datatypes {dict(collections.Counter(x.datatype for x in SAMPLE))} · "
              f"(b,d) pairs {len({N.pair_id(x) for x in SAMPLE})}")

BASE_PATH = os.path.join(CACHE, f'qpatch_baseline_v2__{VARIANT}__{SRC_LANG}-{DST_LANG}.json')   # v2: + IE, logits
base = {(r['key'], r['heads'], r['src_role'], r.get('alpha', 1.0)): r for r in json.load(open(BASE_PATH))} if os.path.exists(BASE_PATH) else {}
A = float(OV_BOOST)
for x in tqdm(SAMPLE, desc='q-patch baseline'):
    need = [(h, r) for h in HEADSETS for r in SRC_ROLES if (x.key, h, r, A) not in base]
    if not need:
        continue
    g = GATE[x.key]; b, d = g['b_id'], g['d_id']
    ps = x.prompts(ctx)
    l_dst = N.last_token_logits(mt, [ps['dst']])[0]
    l_star = N.last_token_logits(mt, [ps['dst_star']])[0]
    lsm_dst = l_dst.log_softmax(-1)
    for hname, role in need:
        lp = N.q_patch_logits(mt, ctx, x, HEADSETS[hname], src_role=role, ov_boost=A)
        lsm = lp.log_softmax(-1)
        base[(x.key, hname, role, A)] = dict(
            key=x.key, datatype=x.datatype, pair=f'{x.pivot.b}->{x.pivot.d}', heads=hname, src_role=role, alpha=A,
            flip=int(lp.argmax()) == d, margin=float(lsm[d] - lsm[b]), clean_margin=float(lsm_dst[d] - lsm_dst[b]),
            ie=N.indirect_effect(lp, l_dst, l_star, d))
    json.dump(list(base.values()), open(BASE_PATH, 'w'))

ALPHAS = sorted({v.get('alpha', 1.0) for v in base.values() if v['key'] in KEYS})
def _rows(h, r, a, dt=None):
    return [v for v in base.values() if v['key'] in KEYS and v['heads'] == h and v['src_role'] == r
            and v.get('alpha', 1.0) == a and (dt is None or v['datatype'] == dt)]
def _ie(rs):
    ies = [r['ie'] for r in rs if r['ie'] == r['ie'] and abs(r['ie']) != float('inf')]
    return (sum(ies) / len(ies), statistics.median(ies)) if ies else (float('nan'), float('nan'))

t = Table('heads', 'query from', 'OV x', 'n', 'flip→d', 'mean Δmargin', 'IE mean', 'IE median', title=f'baseline q-patch · {VARIANT}')
for hname in HEADSETS:
    for role in SRC_ROLES:
      for a in ALPHAS:
        rs = _rows(hname, role, a)
        if rs:
            m, md = _ie(rs)
            t.add_row(hname, role, f'{a:g}', str(len(rs)), f"{sum(r['flip'] for r in rs) / len(rs):.2f}",
                      f"{sum(r['margin'] - r['clean_margin'] for r in rs) / len(rs):+.2f}", f'{m:+.3f}', f'{md:+.3f}')
console.print(t)

t2 = Table('datatype', 'n', *[f'{h}|{r}\nflip / IE med' for h in HEADSETS for r in SRC_ROLES], title=f'by datatype · OV x{A:g}')
for dt in sorted({x.datatype for x in SAMPLE}):
    cells = []
    for h in HEADSETS:
        for r in SRC_ROLES:
            rs = _rows(h, r, A, dt)
            cells.append(f"{sum(x['flip'] for x in rs) / max(1, len(rs)):.2f} / {_ie(rs)[1]:+.2f}" if rs else '—')
    t2.add_row(dt, str(sum(x.datatype == dt for x in SAMPLE)), *cells)
console.print(t2)

sample n=64 · datatypes {'int': 22, 'list': 21, 'str': 21} · (b,d) pairs 48

q-patch baseline:   0%|          | 0/64 [00:00<?, ?it/s]

                          baseline q-patch · order_diff                           
┏━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━┳━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┓
┃ heads   ┃ query from ┃ OV x ┃ n  ┃ flip→d ┃ mean Δmargin ┃ IE mean ┃ IE median ┃
┡━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━╇━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━┩
│ goal250 │ src        │ 2    │ 64 │ 0.25   │ +9.06        │ +0.581  │ +0.523    │
│ goal250 │ src_ctrl   │ 2    │ 64 │ 0.19   │ +6.99        │ +0.374  │ +0.309    │
│ ctrl250 │ src        │ 2    │ 64 │ 0.00   │ +1.01        │ +0.043  │ +0.013    │
│ ctrl250 │ src_ctrl   │ 2    │ 64 │ 0.00   │ +0.55        │ -0.009  │ +0.000    │
└─────────┴────────────┴──────┴────┴────────┴──────────────┴─────────┴───────────┘

                                  by datatype · OV x2                                  
┏━━━━━━━━━━┳━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃          ┃    ┃ goal250|src   ┃ goal250|src_ctrl ┃ ctrl250|src   ┃ ctrl250|src_ctrl ┃
┃ datatype ┃ n  ┃ flip / IE med ┃ flip / IE med    ┃ flip / IE med ┃ flip / IE med    ┃
┡━━━━━━━━━━╇━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ int      │ 22 │ 0.27 / +0.52  │ 0.23 / +0.34     │ 0.00 / +0.01  │ 0.00 / -0.01     │
│ list     │ 21 │ 0.33 / +0.47  │ 0.19 / +0.25     │ 0.00 / +0.06  │ 0.00 / +0.02     │
│ str      │ 21 │ 0.14 / +0.59  │ 0.14 / +0.34     │ 0.00 / -0.03  │ 0.00 / -0.04     │
└──────────┴────┴───────────────┴──────────────────┴───────────────┴──────────────────┘